In [15]:
import sys
sys.path.insert(0, '../..')
from dependencies import *
from constants import *
from paths import *
import helper_functions

## Create gammatone spectrogram

We start by using a gammatone filterbank which mimics the cochlea to create a 128 band gammatone spectrogram.

In [16]:
def compute_gammatone(stimulus):
    """Compute and save gammatone spectrogram for a single stimulus."""
    dst = FUGLSANG_STIMULUS_DIR / f'{stimulus}-gammatone.pickle'

    if dst.exists():
        return

    wav = eelbrain.load.wav(FUGLSANG_STIMULUS_DIR / f'{stimulus}.wav')
    gt  = eelbrain.gammatone_bank(
        wav,
        GAMMATONE_FREQUENCY_RANGE_LOW,
        GAMMATONE_FREQUENCY_RANGE_HIGH,
        GAMMATONE_SPECTROGRAM_BANDS,
        location='left',
        tstep=0.001
    )
    eelbrain.save.pickle(gt, dst)
    print(f"  ✓ Saved {dst}")


def compute_stimulus_predictors(stimulus, padded=False):
    """Compute and save all predictors for a single stimulus."""
    dst_predictor = FUGLSANG_PRED_SELF_DIR / f'{stimulus}~{helper_functions.get_predictor_name(PREDICTOR_TYPE.ENVELOPE, padded)}.pickle'

    if dst_predictor.exists():
        print(f"{stimulus}: predictors exist, skipping.")
        return

    gt     = eelbrain.load.unpickle(FUGLSANG_STIMULUS_DIR / f'{stimulus}-gammatone.pickle')
    gt_log = (gt + 1).log()
    gt_on  = eelbrain.edge_detector(gt_log, c=30)

    predictors = {
        PREDICTOR_TYPE.ENVELOPE:                 gt_log.sum('frequency'),
        PREDICTOR_TYPE.ENVELOPE_ONSET:           gt_on.sum('frequency'),
        PREDICTOR_TYPE.SPECTROGRAM_8_BAND:       gt_log.bin(nbins=8, func='sum', dim='frequency'),
        PREDICTOR_TYPE.SPECTROGRAM_ONSET_8_BAND: gt_on.bin(nbins=8, func='sum', dim='frequency'),
    }

    for predictor_type, predictor in predictors.items():
        p_name = helper_functions.get_predictor_name(predictor_type, padded)
        predictor.name = f'{stimulus}~{p_name}'
        dst = FUGLSANG_PRED_SELF_DIR / f'{stimulus}~{p_name}.pickle'
        eelbrain.save.pickle(helper_functions.process_predictor(predictor, padded), dst)

    print(f"  ✓ Saved {'padded ' if padded else ''}predictors for {stimulus}")

In [17]:
for stimulus in helper_functions.get_stimuli_paths():
    compute_gammatone(stimulus)
    compute_stimulus_predictors(stimulus, padded = True)
    compute_stimulus_predictors(stimulus, padded = False)

print("Done computing stimulus predictors.")

marianne_story5_trial_5: predictors exist, skipping.
marianne_story5_trial_5: predictors exist, skipping.
aske_story2_trial_3: predictors exist, skipping.
aske_story2_trial_3: predictors exist, skipping.
marianne_story4_trial_22: predictors exist, skipping.
marianne_story4_trial_22: predictors exist, skipping.
marianne_story4_trial_23: predictors exist, skipping.
marianne_story4_trial_23: predictors exist, skipping.
aske_story2_trial_2: predictors exist, skipping.
aske_story2_trial_2: predictors exist, skipping.
marianne_story5_trial_4: predictors exist, skipping.
marianne_story5_trial_4: predictors exist, skipping.
marianne_story5_trial_6: predictors exist, skipping.
marianne_story5_trial_6: predictors exist, skipping.
marianne_story4_trial_21: predictors exist, skipping.
marianne_story4_trial_21: predictors exist, skipping.
aske_story2_trial_11: predictors exist, skipping.
aske_story2_trial_11: predictors exist, skipping.
aske_story2_trial_10: predictors exist, skipping.
aske_story2_

# Match predictors to trials and concatenate


In [18]:
helper_functions.get_trials(subject="S1")
# Trials dictionary of type 
# trials = { 
#     0: { 
#         'attended': 'marianne_story3_trial_1', 
#         'ignored': 'aske_story4_trial_1'}
#     }
# }

S1: loaded 60 trials


{0: {'attended': 'marianne_story3_trial_1', 'ignored': 'aske_story4_trial_1'},
 1: {'attended': 'marianne_story3_trial_2', 'ignored': 'aske_story4_trial_2'},
 2: {'attended': 'marianne_story3_trial_3', 'ignored': 'aske_story4_trial_3'},
 3: {'attended': 'marianne_story3_trial_4', 'ignored': 'aske_story4_trial_4'},
 4: {'attended': 'marianne_story3_trial_5', 'ignored': 'aske_story4_trial_5'},
 5: {'attended': 'marianne_story3_trial_6', 'ignored': 'aske_story4_trial_6'},
 6: {'attended': 'marianne_story3_trial_7', 'ignored': 'aske_story4_trial_7'},
 7: {'attended': 'marianne_story3_trial_8', 'ignored': 'aske_story4_trial_8'},
 8: {'attended': 'marianne_story3_trial_9', 'ignored': 'aske_story4_trial_9'},
 9: {'attended': 'marianne_story3_trial_10',
  'ignored': 'aske_story4_trial_10'},
 10: {'attended': 'marianne_story3_trial_11',
  'ignored': 'aske_story6_trial_1'},
 11: {'attended': 'marianne_story3_trial_12',
  'ignored': 'aske_story6_trial_2'},
 12: {'attended': 'marianne_story3_trial

In [19]:
def concatenate_subject_predictors(subject, predictor: PREDICTOR_TYPE, attention: ATTENTION_TYPE, padded=False):
    """
    Concatenate a single predictor across trials for a single subject and attention type,
    following that subject's trial order.

    Args:
        subject:   Subject ID.
        predictor: A single PREDICTOR_TYPE.
        attention: ATTENTION_TYPE.ATTENDED or ATTENTION_TYPE.IGNORED.
        padded:    Whether to load padded predictors.
    """
    trials = helper_functions.get_trials(subject)

    dst_dir = FUGLSANG_PRED_CONCAT_SELF_DIR / subject
    dst_dir.mkdir(exist_ok=True, parents=True)

    name     = helper_functions.get_attentional_predictor_name(predictor, attention, padded)
    out_path = dst_dir / f'{name}_concat.pickle'

    if out_path.exists():
        print(f"{subject}: {name} exists, skipping.")
        return

    p_name        = helper_functions.get_predictor_name(predictor, padded)
    concat_preds = []

    for trial_id, trial in trials.items():
        story = trial[attention.value]
        pred  = eelbrain.load.unpickle(FUGLSANG_PRED_SELF_DIR / f'{story}~{p_name}.pickle')
        concat_preds.append(pred)

    concat = eelbrain.concatenate(concat_preds, name=name)
    eelbrain.save.pickle(concat, out_path)
    print(f"  ✓ Saved {attention.value} → {out_path}")

    return concat

In [20]:
checks = [
    (PREDICTOR_TYPE.ENVELOPE,       ATTENTION_TYPE.ATTENDED, False),
    (PREDICTOR_TYPE.ENVELOPE,       ATTENTION_TYPE.IGNORED,  False),
    (PREDICTOR_TYPE.ENVELOPE,       ATTENTION_TYPE.ATTENDED, True),
    (PREDICTOR_TYPE.ENVELOPE,       ATTENTION_TYPE.IGNORED,  True),
    (PREDICTOR_TYPE.ENVELOPE_ONSET, ATTENTION_TYPE.ATTENDED, False),
    (PREDICTOR_TYPE.ENVELOPE_ONSET, ATTENTION_TYPE.IGNORED,  False),
]

SUBJECTS = helper_functions.fuglsang_get_subjects()
for subject in SUBJECTS:
    for predictor, attention, padded in checks:
        concatenate_subject_predictors(subject, predictor, attention, padded)

print("Done concatenating predictors.")

S1: loaded 60 trials
S1: attended_envelope exists, skipping.
S1: loaded 60 trials
S1: ignored_envelope exists, skipping.
S1: loaded 60 trials
S1: attended_envelope_padded exists, skipping.
S1: loaded 60 trials
S1: ignored_envelope_padded exists, skipping.
S1: loaded 60 trials
S1: attended_envelope_onset exists, skipping.
S1: loaded 60 trials
S1: ignored_envelope_onset exists, skipping.
S2: loaded 60 trials
S2: attended_envelope exists, skipping.
S2: loaded 60 trials
S2: ignored_envelope exists, skipping.
S2: loaded 60 trials
S2: attended_envelope_padded exists, skipping.
S2: loaded 60 trials
S2: ignored_envelope_padded exists, skipping.
S2: loaded 60 trials
S2: attended_envelope_onset exists, skipping.
S2: loaded 60 trials
S2: ignored_envelope_onset exists, skipping.
S3: loaded 60 trials
S3: attended_envelope exists, skipping.
S3: loaded 60 trials
S3: ignored_envelope exists, skipping.
S3: loaded 60 trials
S3: attended_envelope_padded exists, skipping.
S3: loaded 60 trials
S3: ignored_

In [21]:
# SANITY CHECK: Check dimensions of concatenated predictors
subject = SUBJECTS[0]

checks = [
    (PREDICTOR_TYPE.ENVELOPE,       ATTENTION_TYPE.ATTENDED, False),
    (PREDICTOR_TYPE.ENVELOPE,       ATTENTION_TYPE.IGNORED,  False),
    (PREDICTOR_TYPE.ENVELOPE,       ATTENTION_TYPE.ATTENDED, True),
    (PREDICTOR_TYPE.ENVELOPE,       ATTENTION_TYPE.IGNORED,  True),
    (PREDICTOR_TYPE.ENVELOPE_ONSET, ATTENTION_TYPE.ATTENDED, False),
    (PREDICTOR_TYPE.ENVELOPE_ONSET, ATTENTION_TYPE.IGNORED,  False),
]

for predictor, attention, padded in checks:
    name = helper_functions.get_attentional_predictor_name(predictor, attention, padded)
    path = FUGLSANG_PRED_CONCAT_SELF_DIR / subject / f'{name}_concat.pickle'
    if path.exists():
        print(f"  ✓ {name}: {eelbrain.load.unpickle(path)}")
    else:
        print(f"  ✗ {name}: MISSING")

  ✓ attended_envelope: <NDVar 'attended_envelope': 192000 time>
  ✓ ignored_envelope: <NDVar 'ignored_envelope': 192000 time>
  ✓ attended_envelope_padded: <NDVar 'attended_envelope_padded': 196260 time>
  ✓ ignored_envelope_padded: <NDVar 'ignored_envelope_padded': 196260 time>
  ✓ attended_envelope_onset: <NDVar 'attended_envelope_onset': 192000 time>
  ✓ ignored_envelope_onset: <NDVar 'ignored_envelope_onset': 192000 time>
